<a href="https://colab.research.google.com/github/Alex-Roberts27/Research_Fibrosarcoma_OAS_Plots/blob/main/OAS_Interferon_Genes_EW46vMock_Plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We took HT-1080 fibrosarcoma cells and infected them with either DB62 (infects well and kills cells) or EW46 (infects a lot, but doesn't kill). We then removed the media and lysed/homogenized in 200 µl/well of Zymo DNA/RNA shield and stored them at -20 °C. We sent them for whole-cell RNA-sequencing to Plasmidsaurus. We want to see which genes are upregulated or downregulated. Our samples are:


*   Samples 1 and 2: HT-1080_Mock 1, HT-1080_Mock 2 (two biological replicates,
mock-treated cells)
*   Samples 3 and 4: HT-1080_DB62 1, HT-1080_DB62 2 (two biological replicates, DB62-infected cells)
*   Samples 5 and 6: HT-1080_EW46 1, HT-1080_EW46 2 (two biological replicates, EW46-infected cells)

Code for Heatmaps came from:
1. https://github.com/mousepixels/sanbomics_scripts/blob/main/PyDeseq2_DE_tutorial.ipynb

Code for Volcano Plots came from:
1. https://www.youtube.com/watch?v=qHeIhEPKU98

This File explores ISG (Interferon Stimulated Genes and their expression in the samples). Specifically OAS genes.

# Load Packages and RNAseq File

In [ ]:
%pip install --upgrade typing-extensions
%pip uninstall anndata -y
%pip uninstall pydeseq2 -y
%pip install pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd

In [ ]:
# Load the Excel file
df = pd.read_excel('RNA-seq-matrix-for-efra.xlsx')

# Convert and save as CSV
# index=False prevents pandas from adding an extra column for row numbers
df.to_csv('RNA-seq-matrix-for-efra.csv', index=False)

In [ ]:
counts = pd.read_csv('RNA-seq-matrix-for-efra.csv')
counts

In [ ]:
#If the gene_biotype is not protein_coding, remove it from the sheet
counts = counts[counts.gene_biotype == 'protein_coding']
counts

In [ ]:
#Remove the gene_name column
counts = counts.drop(columns=['gene_name'])
#Remove the gene_biotype column
counts = counts.drop(columns=['gene_biotype'])
counts

In [ ]:
#Makes final counts dataframe with raw gene counts by samples.
columns_to_drop = [
    'HT-1080_Mock_1_cpm',
    'HT-1080_Mock_2_cpm',
    'HT-1080_DB62_1_cpm',
    'HT-1080_DB62_2_cpm',
    'HT-1080_EW46_1_cpm',
    'HT-1080_EW46_2_cpm'
]
counts = counts.drop(columns=columns_to_drop)
counts

In [ ]:
counts = counts.set_index('gene_id')
counts

# Filter and clean Counts Dataframe along with deseq analysis on whole dataframe

In [ ]:
#Selects for only numeric rows, removes rows where NA is present since they can't be used in ddseq
numeric_columns = counts.select_dtypes(include=['number']).columns
counts = counts[counts[numeric_columns].sum(axis=1) > 0]
counts

In [ ]:
#Can change counts filter. Currently at 5, but can be from 1 to 10 in an acceptable way.
  #Selects for genes where a raw count is observed and a significant enough change is observed between samples
sig_counts = counts[~(counts < 5).all(axis=1)]
sig_counts

In [ ]:
#Rename Columns HT-1080_Mock_1_count to Mock_1
sig_counts = sig_counts.rename(columns={'HT-1080_Mock_1_count': 'Mock_1',
                                        'HT-1080_Mock_2_count': 'Mock_2',
                                        'HT-1080_DB62_1_count': 'DB62_1',
                                        'HT-1080_DB62_2_count': 'DB62_2',
                                        'HT-1080_EW46_1_count': 'EW46_1',
                                        'HT-1080_EW46_2_count': 'EW46_2'})
sig_counts

In [ ]:
#Checks to see if any NA raw count colums exist
sig_counts.isnull().sum()

In [ ]:
flippedcounts = sig_counts.T
flippedcounts

In [ ]:
#Index part is relevant to the samples the counts dataframe has, they are changeable
metadata = pd.DataFrame(zip(flippedcounts.index, ['Mock','Mock','DB62','DB62', 'EW46', 'EW46']),
                        columns = ['Sample', 'Condition'])

In [ ]:
metadata = metadata.set_index('Sample')
metadata

In [ ]:
flippedcounts = flippedcounts.round().astype(int)
dds = DeseqDataSet(counts=flippedcounts,
            metadata=metadata,
            design="~Condition")

In [ ]:
dds.deseq2()

In [ ]:
dds

# Making non-filtered gene dataframe for plots

In [ ]:
# Example: Comparing EW46 vs Mock
stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','EW46','Mock'))

#Uses DB62 as the control, so when using them vs mock do (DB62, mock) and (EW46, mock)

In [ ]:
stat_res.summary()

In [ ]:
res = stat_res.results_df

In [ ]:
res

In [ ]:
#Drops rows where deseq determined gene wasn't relevant, i.e. NA
res = res.dropna(how='any')
res

In [ ]:
!pip install sanbomics

In [ ]:
from sanbomics.tools import id_map

In [ ]:
mapper = id_map(species = "human")
mapper.mapper

In [ ]:
res["Symbol"] = res.index.map(mapper.mapper)

In [ ]:
res

In [ ]:
#res dataframe to a csv file
#res.to_csv('fullgenesetDB62vEW46.csv')
#only for a volcano plot if we need to make a full one without having preselected genes

# Finding ISG Genes in Res Dataframe and filtering for them only

In [ ]:
# Load the Excel file
# Use sheet_name=None to load all sheets, or specify one (e.g., sheet_name=0)
df = pd.read_excel('ISG_Table.xlsx')

# Convert and save as CSV
# index=False prevents pandas from adding an extra column for row numbers
df.to_csv('ISG_Table.csv', index=False)

In [ ]:
isgset = pd.read_csv('ISG_Table.csv')
isgset

In [ ]:
#Extract list of Genes from isgset as a string
isglist = isgset['Genes'].tolist()
isglist

In [ ]:
#Search res dataframe for genes in isglist and build dataframe with only those genes
isg_res = res[res['Symbol'].isin(isglist)]
isg_res

# Filtering down gene dataframe to only significant genes, can't use it for volcano plot

In [ ]:
sigs = isg_res[(isg_res.padj < 0.05) & (abs(isg_res.log2FoldChange) > 1)]
#Selects for only significantly different genes by adjp-value and log2fold change.
#See https://pmc.ncbi.nlm.nih.gov/articles/PMC4670015/ for what values to select at.

In [ ]:
sigs

In [ ]:
#Remove rows where symbol is NaN
cleanedsigs = sigs.dropna(subset=['Symbol'])
cleanedsigs #Final dataframe for heatmaps

In [ ]:
cleanedsigs.info()

# Volcano with all genes

In [ ]:
isg_res

In [ ]:
res2 = isg_res
res_no_na = res2.dropna(how='any')
res_no_na

In [ ]:
#Have to make the dataframe into a csv file to interact with gene_id
res_no_na.to_csv('resnona.csv')

## Where volcano plot code starts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
volcanononsig = pd.read_csv('resnona.csv')

In [ ]:
#Drop columns not needed
volcanononsig1 = volcanononsig.drop(['gene_id', 'baseMean', 'lfcSE', 'stat', 'padj'], axis=1)
volcanononsigfull = volcanononsig1.rename(columns={'Symbol': 'Gene'})
volcanononsigfullfinal = volcanononsigfull[['Gene', 'log2FoldChange', 'pvalue']].copy()
volcanononsigfullfinal #Final dataframe used for generating full volcano plot

In [ ]:
significant_genes = (volcanononsigfullfinal["pvalue"] < 0.05) & (abs(volcanononsigfullfinal["log2FoldChange"]) > 1)

# Calculate NegLog10Pvalue
volcanononsigfullfinal["NegLog10Pvalue"] = -np.log10(volcanononsigfullfinal["pvalue"])

# Assign regulation status based on criteria
volcanononsigfullfinal["Regulation"] = "Not Significant" # Default
volcanononsigfullfinal.loc[(volcanononsigfullfinal["pvalue"] < 0.05) & (volcanononsigfullfinal["log2FoldChange"] > 1), "Regulation"] = "Upregulated"
volcanononsigfullfinal.loc[(volcanononsigfullfinal["pvalue"] < 0.05) & (volcanononsigfullfinal["log2FoldChange"] < -1), "Regulation"] = "Downregulated"

# Define the palette to map regulation labels to colors
regulation_palette = {
    "Upregulated": "red",
    "Downregulated": "blue",
    "Not Significant": "gray"
}

#Set figure size and plot
plt.figure(figsize=(8,6), dpi=300)
sns.scatterplot(data=volcanononsigfullfinal, x="log2FoldChange", y="NegLog10Pvalue", hue="Regulation", palette=regulation_palette)
plt.xlabel("Log2 Fold Change", fontsize=12)
plt.ylabel("-Log10 P-Value", fontsize=12)
plt.title("Volcano Plot of RNA-Seq Data EW46 vs Mock", fontsize=14, fontweight='bold')
plt.axhline(y=1, linestyle='--', color='red')
plt.axvline(x=1, linestyle='--', color='blue')
plt.axvline(x=-1, linestyle='--', color='blue')
# Let seaborn handle the legend labels automatically from the 'Regulation' column
plt.legend(title="Gene Regulation Mock v EW46", loc="upper right")
plt.savefig("volcano_plotDB62vEW46.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
significant_genes = (volcanononsigfullfinal["pvalue"] < 0.05) & (abs(volcanononsigfullfinal["log2FoldChange"]) > 1)

# Calculate NegLog10Pvalue
volcanononsigfullfinal["NegLog10Pvalue"] = -np.log10(volcanononsigfullfinal["pvalue"])

# Assign regulation status based on criteria
volcanononsigfullfinal["Regulation"] = "Not Significant" # Default
volcanononsigfullfinal.loc[(volcanononsigfullfinal["pvalue"] < 0.05) & (volcanononsigfullfinal["log2FoldChange"] > 1), "Regulation"] = "Upregulated"
volcanononsigfullfinal.loc[(volcanononsigfullfinal["pvalue"] < 0.05) & (volcanononsigfullfinal["log2FoldChange"] < -1), "Regulation"] = "Downregulated"

# Define the palette to map regulation labels to colors
regulation_palette = {
    "Upregulated": "red",
    "Downregulated": "blue",
    "Not Significant": "gray"
}

#Set figure size and plot
plt.figure(figsize=(8,6), dpi=300)
sns.scatterplot(data=volcanononsigfullfinal, x="log2FoldChange", y="NegLog10Pvalue", hue="Regulation", palette=regulation_palette)
plt.xlabel("Log2 Fold Change", fontsize=12)
plt.ylabel("-Log10 P-Value", fontsize=12)
plt.title("Volcano Plot of RNA-Seq Data EW46 vs Mock", fontsize=14, fontweight='bold')
plt.axhline(y=1, linestyle='--', color='red')
plt.axvline(x=1, linestyle='--', color='blue')
plt.axvline(x=-1, linestyle='--', color='blue')
# Let seaborn handle the legend labels automatically from the 'Regulation' column
plt.legend(loc="upper left")
#Set Horizontal Axis to be Equal
max_abs_log2FoldChange = (volcanononsigfullfinal['log2FoldChange'].abs().max() + 0.1)
plt.xlim(-max_abs_log2FoldChange, max_abs_log2FoldChange)
plt.xticks(np.arange(int(-max_abs_log2FoldChange), int(max_abs_log2FoldChange) + 1, 1))
max_neglog10pvalue = volcanononsigfullfinal['NegLog10Pvalue'].max()
plt.yticks(np.arange(0, int(max_neglog10pvalue) + 20, 20))
plt.show()

In [ ]:
from adjustText import adjust_text

In [ ]:
fig, ax = plt.figure(figsize=(10,8), dpi=300), plt.gca()
sns.scatterplot(data=volcanononsigfullfinal, x="log2FoldChange", y="NegLog10Pvalue", hue="Regulation", palette=regulation_palette, ax=ax)

plt.xlabel("Log2 Fold Change", fontsize=12)
plt.ylabel("-Log10 P-Value", fontsize=12)
plt.title("Volcano Plot of RNA-Seq Data EW46 vs Mock with Interferon Genes", fontsize=14, fontweight='bold')
plt.axhline(y=1, linestyle='--', color='red')
plt.axvline(x=1, linestyle='--', color='blue')
plt.axvline(x=-1, linestyle='--', color='blue')
plt.legend(loc="upper left")

# Filter for significant genes with 'Upregulated' or 'Downregulated' status
significant_labeled_genes = volcanononsigfullfinal[(volcanononsigfullfinal['Regulation'] == 'Upregulated') | (volcanononsigfullfinal['Regulation'] == 'Downregulated')]

# Create list of texts to adjust
texts = []
for i, row in significant_labeled_genes.iterrows():
    texts.append(plt.text(row['log2FoldChange'], row['NegLog10Pvalue'], row['Gene'], fontsize=8))

# Adjust text to prevent overlapping
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
max_abs_log2FoldChange = (volcanononsigfullfinal['log2FoldChange'].abs().max() + 0.1)

# Set x-axis ticks to increase by 1
plt.xlim(-max_abs_log2FoldChange, max_abs_log2FoldChange)
plt.xticks(np.arange(int(-max_abs_log2FoldChange), int(max_abs_log2FoldChange) + 1, 1))

# Set y-axis ticks to increase by 20
max_neglog10pvalue = volcanononsigfullfinal['NegLog10Pvalue'].max()
plt.yticks(np.arange(0, int(max_neglog10pvalue) + 20, 20))

plt.savefig("volcano_plot_labeled_DB62vEW46.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
#This volcano plot shows EW46 genes and if they are upregulated or downregulated compared to Mock.

# Heatmap with only significant genes

In [ ]:
dds.layers["normed_counts"]

In [ ]:
dds.layers["log1p"] = np.log1p(dds.layers["normed_counts"])

In [ ]:
dds.layers["log1p"]

In [ ]:
#This is the dataframe where we'll use all the remaining genes
cleanedsigs

In [ ]:
#filter for OASL, OAS2, OAS3, OAS1 genes
oas_genes = ['OASL', 'OAS2', 'OAS3', 'OAS1']
oas_sigs = cleanedsigs[cleanedsigs['Symbol'].isin(oas_genes)]
oas_sigs

In [ ]:
dds_sigs1 = dds[:,oas_sigs.index]
dds_sigs1

In [ ]:
heatmapfull = pd.DataFrame(dds_sigs1.layers["log1p"].T, index=dds_sigs1.var_names, columns=dds_sigs1.obs_names)

In [ ]:
sns.clustermap(heatmapfull, z_score=0, cmap = "coolwarm")

In [ ]:
heatmapfull.index = heatmapfull.index.map(mapper.mapper)

In [ ]:
#if you want to make a CSV list of genes
#heatmapfull
#heatmapfull.to_csv('heatmap.csv')

In [ ]:
labeledheatmapfull = sns.clustermap(heatmapfull, z_score=0, cmap = "coolwarm", figsize=(4,6))
'''EW46 v Mock'''

# Top 24 significant genes Heatmap

In [ ]:
# Keep top 24 genes by lowest and highest log2foldchanges
top_12_lowest_lfc = cleanedsigs.nsmallest(12, 'log2FoldChange')
top_12_highest_lfc = cleanedsigs.nlargest(12, 'log2FoldChange')
top_24_genes_for_plots = pd.concat([top_12_lowest_lfc, top_12_highest_lfc])
top_24_genes_for_plots

In [ ]:
dds_sigs = dds[:,top_24_genes_for_plots.index]
dds_sigs

In [ ]:
smallheatmap = pd.DataFrame(dds_sigs.layers["log1p"].T, index=dds_sigs.var_names, columns=dds_sigs.obs_names)

In [ ]:
sns.clustermap(smallheatmap, z_score=0, cmap = "coolwarm")

In [ ]:
smallheatmap.index = smallheatmap.index.map(mapper.mapper)

In [ ]:
#smallheatmap
#smallheatmap.to_csv('smallheatmap.csv')
#again only neccesary if you want a csv

In [ ]:
labeledheatmapsmall = sns.clustermap(smallheatmap, z_score=0, cmap = "coolwarm", figsize=(4,8))
'''EW46 v Mock'''

## Ranking by absolute log2fold change to see if a diff appears

In [ ]:
cleanedsigs['abs_log2FoldChange'] = cleanedsigs['log2FoldChange'].abs()
top_24_genes_by_abs_lfc = cleanedsigs.nlargest(24, 'abs_log2FoldChange')
top_24_genes_by_abs_lfc

In [ ]:
dds_sigs = dds[:,top_24_genes_by_abs_lfc.index]
dds_sigs

In [ ]:
rankedsmallheatmap = pd.DataFrame(dds_sigs.layers["log1p"].T, index=dds_sigs.var_names, columns=dds_sigs.obs_names)

In [ ]:
sns.clustermap(rankedsmallheatmap, z_score=0, cmap = "coolwarm")

In [ ]:
rankedsmallheatmap.index = rankedsmallheatmap.index.map(mapper.mapper)

In [ ]:
ranklabeledheatmapsmall = sns.clustermap(rankedsmallheatmap, z_score=0, cmap = "coolwarm", figsize=(4,8))
'''EW46 v Mock'''